# 🧪 Modul Eksperimen: Advanced ConvNeXt (384px)
Dipanggil dari `bdc_colab_t4_fixed.ipynb` menggunakan `%run bdc_training_advance.ipynb`

## 1. Setup & Import

In [ ]:
import os, warnings, random, shutil
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, datasets
import timm
from PIL import Image
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import f1_score, classification_report
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('[Advance] Device:', device)
if device.type == 'cuda':
    print('[Advance] GPU   :', torch.cuda.get_device_name(0))

## 2. Directory Setup

In [ ]:
os.makedirs('models', exist_ok=True)
os.makedirs('submission', exist_ok=True)

DRIVE_SAVE_DIR = '/content/drive/MyDrive/BDC/models'
if os.path.exists('/content/drive/MyDrive'):
    os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

print('[Advance] Folder Ready')

## 3. Transforms (384px)

In [ ]:
IMG_SIZE = 384

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(40),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE + 32),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
print('[Advance] Transforms 384px siap!')

## 4. Load Dataset & DataLoader

In [ ]:
train_dir = '/content/proyek_bdc/data/train' if os.path.exists('/content/proyek_bdc/data/train') else 'data/train'
full_dataset = datasets.ImageFolder(root=train_dir, transform=train_transform)

train_size = int(0.8 * len(full_dataset))
val_size   = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(
    full_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

val_dataset.dataset.transform = val_transform   

train_loader = DataLoader(train_dataset, batch_size=24, shuffle=True, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

print('[Advance] Class mapping:', full_dataset.class_to_idx)
print(f'[Advance] Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}')

## 5. Model ConvNeXt Tiny

In [ ]:
model_name = 'convnext_tiny'
model = timm.create_model(model_name, pretrained=True, num_classes=3, drop_path_rate=0.2)
model = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
print('[Advance] Model & Optimizer siap!')

## 6. Training Loop (15 Epochs)

In [ ]:
best_f1 = 0.0
num_epochs = 15
LAST_CKPT_PATH = os.path.join(DRIVE_SAVE_DIR, 'last_convnext_advance.pth')
BEST_CKPT_PATH = os.path.join(DRIVE_SAVE_DIR, 'best_convnext_advance.pth')

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    pbar = tqdm(train_loader, desc=f'Advance Epoch {epoch+1}/{num_epochs} [Train]')
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f'Epoch {epoch+1} [Val]', leave=False):
            images = images.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    
    val_f1 = f1_score(all_labels, all_preds, average='macro')
    print(f'\n[Advance] Epoch {epoch+1} - Loss: {train_loss/len(train_loader):.4f} | Val Macro F1: {val_f1:.4f}')
    
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), 'models/best_convnext_advance.pth')
        if os.path.exists('/content/drive/MyDrive'):
            torch.save(model.state_dict(), BEST_CKPT_PATH)
        print(f'   [BEST] F1 baru = {best_f1:.4f}')
    
    scheduler.step()

print(f'\n[Advance] Training Selesai! Best Macro F1: {best_f1:.4f}')

## 7. Test Prediction & Submission

In [ ]:
class TestDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_files = sorted([f for f in os.listdir(root_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))])
    
    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, self.image_files[idx])
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, self.image_files[idx]

test_dir = '/content/proyek_bdc/data/test' if os.path.exists('/content/proyek_bdc/data/test') else 'data/test'
test_dataset = TestDataset(test_dir, transform=val_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=False)

model.load_state_dict(torch.load('models/best_convnext_advance.pth', weights_only=True))
model.eval()

submission = []
with torch.no_grad():
    for images, filenames in tqdm(test_loader, desc='Predicting'):
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()
        for fname, p in zip(filenames, preds):
            submission.append({'filename': fname, 'predicted': int(p)})

sub_df = pd.DataFrame(submission)
sub_df.insert(0, 'id', range(len(sub_df)))
sub_df.to_csv('submission/submission_advance.csv', index=False)
if os.path.exists('/content/drive/MyDrive'):
    shutil.copy('submission/submission_advance.csv', os.path.join(DRIVE_SAVE_DIR, 'submission_advance.csv'))
print('[Advance] Submission selesai! Total:', len(sub_df))
print(sub_df.head())